# Single Stock Investment Story Prototype

This notebook is the first draft of the app's calculation brain. It keeps the functions here on purpose so we can inspect the math, tune the story, and only then move stable pieces into `analysis_engine`.

Core question: if someone invested a starting amount in one stock, what happened versus SPY, inflation, and the company's own event timeline?

In [ ]:
from __future__ import annotations

import json
import math
import os
from dataclasses import dataclass
from datetime import date, datetime
from pathlib import Path
from typing import Any, Optional, Union

os.environ.setdefault("MPLCONFIGDIR", str(Path("/tmp") / "matplotlib-cache"))

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
plt.style.use("seaborn-v0_8-whitegrid")


## Inputs

Change these values to test a different story. Start with one ticker and one benchmark so we can make the results feel right before generalizing.

In [ ]:
TICKER = "AAPL"
BENCHMARK = "SPY"
START_DATE = "2015-01-02"
END_DATE = None  # None means use latest available market date shared by ticker and benchmark.
INITIAL_AMOUNT = 10_000
CURRENCY = "USD"

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DATA_DIR = PROJECT_ROOT / "public" / "data"

assert DATA_DIR.exists(), f"Could not find data directory: {DATA_DIR}"
DATA_DIR


## Data Loading Helpers

These are intentionally simple and local to the notebook. Later, the stable versions can become `analysis_engine` functions that write cached JSON.

In [ ]:
def read_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def load_price_frame(ticker: str) -> pd.DataFrame:
    payload = read_json(DATA_DIR / "prices" / f"{ticker}.json")
    df = pd.DataFrame(payload["prices"])
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)
    df["ticker"] = ticker
    return df


def load_events(ticker: str) -> dict[str, Any]:
    path = DATA_DIR / "events" / f"{ticker}.json"
    return read_json(path) if path.exists() else {"timeline_events": [], "recent_news": []}


def load_inflation(currency: str = "USD") -> pd.DataFrame:
    payload = read_json(DATA_DIR / "inflation" / "inflation.json")
    rows = payload["inflation"][currency]
    df = pd.DataFrame(rows).sort_values("year").reset_index(drop=True)
    df["currency"] = currency
    return df


prices = load_price_frame(TICKER)
benchmark_prices = load_price_frame(BENCHMARK)
events = load_events(TICKER)
inflation = load_inflation(CURRENCY)

prices.head(3), prices.tail(3), inflation.tail(5)


## Core Calculation Functions

First version of the math: nearest trading dates, total return, benchmark comparison, inflation adjustment, drawdown, and best/worst rolling periods.

In [ ]:
def as_timestamp(value: Optional[str]) -> Optional[pd.Timestamp]:
    return None if value is None else pd.Timestamp(value)


def row_on_or_after(df: pd.DataFrame, value: Union[str, pd.Timestamp]) -> pd.Series:
    target = pd.Timestamp(value)
    rows = df[df["date"] >= target]
    if rows.empty:
        raise ValueError(f"No row on or after {target.date()}")
    return rows.iloc[0]


def row_on_or_before(df: pd.DataFrame, value: Union[str, pd.Timestamp]) -> pd.Series:
    target = pd.Timestamp(value)
    rows = df[df["date"] <= target]
    if rows.empty:
        raise ValueError(f"No row on or before {target.date()}")
    return rows.iloc[-1]


def shared_end_date(*frames: pd.DataFrame) -> pd.Timestamp:
    return min(frame["date"].max() for frame in frames)


def investment_result(df: pd.DataFrame, start_date: str, end_date: Optional[str], initial_amount: float) -> dict[str, Any]:
    actual_end = as_timestamp(end_date) or df["date"].max()
    start = row_on_or_after(df, start_date)
    end = row_on_or_before(df, actual_end)
    start_price = float(start["adj_close"])
    end_price = float(end["adj_close"])
    shares = initial_amount / start_price
    final_value = shares * end_price
    total_return = final_value / initial_amount - 1
    return {
        "ticker": str(df["ticker"].iloc[0]),
        "requested_start_date": start_date,
        "actual_start_date": start["date"].date().isoformat(),
        "actual_end_date": end["date"].date().isoformat(),
        "initial_amount": initial_amount,
        "start_price": start_price,
        "end_price": end_price,
        "shares": shares,
        "final_value": final_value,
        "profit": final_value - initial_amount,
        "total_return": total_return,
    }


def slice_period(df: pd.DataFrame, start_date: str, end_date: str) -> pd.DataFrame:
    start = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)
    return df[(df["date"] >= start) & (df["date"] <= end)].copy().reset_index(drop=True)


In [ ]:
def cpi_for_year(inflation_df: pd.DataFrame, year: int) -> float:
    available = inflation_df[inflation_df["year"] <= year]
    if available.empty:
        return float(inflation_df.iloc[0]["cpi"])
    # Inflation data currently ends at 2024. For later dates, use latest CPI as a conservative placeholder.
    return float(available.iloc[-1]["cpi"])


def inflation_adjustment(inflation_df: pd.DataFrame, nominal_value: float, start_date: str, end_date: str) -> dict[str, Any]:
    start_year = pd.Timestamp(start_date).year
    end_year = pd.Timestamp(end_date).year
    start_cpi = cpi_for_year(inflation_df, start_year)
    end_cpi = cpi_for_year(inflation_df, end_year)
    factor = end_cpi / start_cpi if start_cpi else 1
    real_final_value = nominal_value / factor if factor else nominal_value
    return {
        "currency": str(inflation_df["currency"].iloc[0]),
        "start_year": start_year,
        "end_year": end_year,
        "start_cpi": start_cpi,
        "end_cpi": end_cpi,
        "inflation_factor": factor,
        "real_final_value_in_start_year_dollars": real_final_value,
        "note": "Uses latest available CPI for years after the inflation dataset ends.",
    }


def drawdown_stats(df: pd.DataFrame, start_date: str, end_date: str) -> dict[str, Any]:
    period = slice_period(df, start_date, end_date)
    period["wealth_index"] = period["adj_close"] / period["adj_close"].iloc[0]
    period["running_peak"] = period["wealth_index"].cummax()
    period["drawdown"] = period["wealth_index"] / period["running_peak"] - 1
    trough = period.loc[period["drawdown"].idxmin()]
    peak_before = period.loc[: trough.name]
    peak = peak_before.loc[peak_before["wealth_index"].idxmax()]
    recovered = period[(period["date"] > trough["date"]) & (period["wealth_index"] >= peak["wealth_index"])]
    recovery_date = None if recovered.empty else recovered.iloc[0]["date"].date().isoformat()
    return {
        "max_drawdown": float(trough["drawdown"]),
        "peak_date": peak["date"].date().isoformat(),
        "trough_date": trough["date"].date().isoformat(),
        "recovery_date": recovery_date,
    }


def best_worst_rolling_return(df: pd.DataFrame, start_date: str, end_date: str, window_days: int = 252) -> dict[str, Any]:
    period = slice_period(df, start_date, end_date)
    period["rolling_return"] = period["adj_close"] / period["adj_close"].shift(window_days) - 1
    valid = period.dropna(subset=["rolling_return"])
    if valid.empty:
        return {"window_trading_days": window_days, "best": None, "worst": None}
    best = valid.loc[valid["rolling_return"].idxmax()]
    worst = valid.loc[valid["rolling_return"].idxmin()]
    return {
        "window_trading_days": window_days,
        "best": {"end_date": best["date"].date().isoformat(), "return": float(best["rolling_return"])},
        "worst": {"end_date": worst["date"].date().isoformat(), "return": float(worst["rolling_return"])},
    }


## Event Selection

For the prototype, we include major timeline events inside the investment window and recent news from the source file. Later, we can connect events to price jumps and drawdowns.

In [ ]:
def events_in_period(event_payload: dict[str, Any], start_date: str, end_date: str, limit: int = 12) -> pd.DataFrame:
    rows = event_payload.get("timeline_events", [])
    if not rows:
        return pd.DataFrame(columns=["date", "event_type", "description"])
    df = pd.DataFrame(rows)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])
    df = df[(df["date"] >= pd.Timestamp(start_date)) & (df["date"] <= pd.Timestamp(end_date))]
    priority = {
        "Earnings Announcement": 1,
        "Stock Split": 2,
        "Dividend Payment": 3,
    }
    df["priority"] = df["event_type"].map(priority).fillna(9)
    df = df.sort_values(["priority", "date"], ascending=[True, False]).head(limit)
    return df[["date", "event_type", "description"]].sort_values("date")


def ceo_events_in_period(event_payload: dict[str, Any], start_date: str, end_date: str, limit: int = 8) -> pd.DataFrame:
    rows = event_payload.get("timeline_events", [])
    if not rows:
        return pd.DataFrame(columns=["date", "event_type", "description", "executive_name", "role", "action"])
    df = pd.DataFrame(rows)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])
    details = df["details"].apply(lambda value: value if isinstance(value, dict) else {})
    df["executive_name"] = details.apply(lambda value: value.get("executive_name"))
    df["role"] = details.apply(lambda value: value.get("role", ""))
    df["action"] = details.apply(lambda value: value.get("action", ""))
    df["source"] = details.apply(lambda value: value.get("source", ""))
    text = (df["description"].fillna("") + " " + df["role"].fillna("")).str.lower()
    ceo_mask = text.str.contains("chief executive officer|\\bceo\\b", regex=True)
    df = df[ceo_mask]
    df = df[(df["date"] >= pd.Timestamp(start_date)) & (df["date"] <= pd.Timestamp(end_date))]
    if df.empty:
        return pd.DataFrame(columns=["date", "event_type", "description", "executive_name", "role", "action"])
    df = df.sort_values("date", ascending=False).head(limit)
    return df[["date", "event_type", "description", "executive_name", "role", "action"]].sort_values("date")


def recent_news_table(event_payload: dict[str, Any], limit: int = 5) -> pd.DataFrame:
    rows = event_payload.get("recent_news", [])[:limit]
    if not rows:
        return pd.DataFrame(columns=["published_at", "publisher", "title", "link"])
    df = pd.DataFrame(rows)
    return df[["published_at", "publisher", "title", "link"]]


## Build One Story Result

This combines the calculation blocks into one object. That object is a preview of the JSON shape we may later cache under `public/data/calculated`.

In [ ]:
actual_end = (as_timestamp(END_DATE) or shared_end_date(prices, benchmark_prices)).date().isoformat()

stock_result = investment_result(prices, START_DATE, actual_end, INITIAL_AMOUNT)
benchmark_result = investment_result(benchmark_prices, START_DATE, actual_end, INITIAL_AMOUNT)
real = inflation_adjustment(inflation, stock_result["final_value"], stock_result["actual_start_date"], stock_result["actual_end_date"])
drawdown = drawdown_stats(prices, stock_result["actual_start_date"], stock_result["actual_end_date"])
rolling = best_worst_rolling_return(prices, stock_result["actual_start_date"], stock_result["actual_end_date"])

story_result = {
    "input": {
        "ticker": TICKER,
        "benchmark": BENCHMARK,
        "start_date": START_DATE,
        "end_date": actual_end,
        "initial_amount": INITIAL_AMOUNT,
        "currency": CURRENCY,
    },
    "stock": stock_result,
    "benchmark": benchmark_result,
    "comparison": {
        "final_value_difference": stock_result["final_value"] - benchmark_result["final_value"],
        "return_difference_pct_points": (stock_result["total_return"] - benchmark_result["total_return"]) * 100,
        "stock_to_benchmark_multiple": stock_result["final_value"] / benchmark_result["final_value"],
    },
    "inflation": real,
    "risk_journey": {
        "drawdown": drawdown,
        "rolling_12m": rolling,
    },
}

story_result


## Human-Readable Summary

This is the first storytelling layer. It should be direct, factual, and emotionally honest about the path, not just the endpoint.

In [ ]:
def money(value: float, currency: str = "USD") -> str:
    symbol = "$" if currency == "USD" else f"{currency} "
    return f"{symbol}{value:,.0f}"


def pct(value: float) -> str:
    return f"{value * 100:,.1f}%"


def write_story(result: dict[str, Any]) -> str:
    stock = result["stock"]
    bench = result["benchmark"]
    comp = result["comparison"]
    infl = result["inflation"]
    dd = result["risk_journey"]["drawdown"]
    rolling = result["risk_journey"]["rolling_12m"]
    currency = result["input"]["currency"]
    ticker = result["input"]["ticker"]
    benchmark = result["input"]["benchmark"]

    best_line = ""
    worst_line = ""
    if rolling["best"]:
        best_line = f" Its strongest roughly 12-month stretch ended on {rolling['best']['end_date']}, returning {pct(rolling['best']['return'])}."
    if rolling["worst"]:
        worst_line = f" Its hardest roughly 12-month stretch ended on {rolling['worst']['end_date']}, returning {pct(rolling['worst']['return'])}."

    recovery = dd["recovery_date"] or "not recovered within this selected window"
    return (
        f"A {money(stock['initial_amount'], currency)} investment in {ticker} on {stock['actual_start_date']} "
        f"would be worth {money(stock['final_value'], currency)} by {stock['actual_end_date']}. "
        f"That is a total return of {pct(stock['total_return'])}, or {money(stock['profit'], currency)} in profit.\n\n"
        f"The same investment in {benchmark} would be worth {money(bench['final_value'], currency)}, "
        f"so {ticker} finished ahead by {money(comp['final_value_difference'], currency)}. "
        f"In return terms, that is a {comp['return_difference_pct_points']:,.1f} percentage-point gap.\n\n"
        f"The journey was not smooth. The largest drawdown was {pct(dd['max_drawdown'])}, "
        f"from a peak on {dd['peak_date']} to a trough on {dd['trough_date']}; recovery date: {recovery}."
        f"{best_line}{worst_line}\n\n"
        f"After inflation, the final value is roughly {money(infl['real_final_value_in_start_year_dollars'], currency)} "
        f"in {infl['start_year']} purchasing-power terms. {infl['note']}"
    )


print(write_story(story_result))


## Story View Prototype

These charts are meant to resemble the actual app experience: a focused visual, a short explanation, and enough annotations to explain the investment journey without making the user inspect a table.

In [ ]:
def wealth_frame(df: pd.DataFrame, start_date: str, end_date: str, initial_amount: float) -> pd.DataFrame:
    period = slice_period(df, start_date, end_date)
    start_price = row_on_or_after(period, start_date)["adj_close"]
    period["wealth"] = initial_amount * period["adj_close"] / start_price
    period["return"] = period["wealth"] / initial_amount - 1
    return period[["date", "ticker", "adj_close", "wealth", "return"]]


def drawdown_frame(df: pd.DataFrame, start_date: str, end_date: str) -> pd.DataFrame:
    period = slice_period(df, start_date, end_date)
    period["wealth_index"] = period["adj_close"] / period["adj_close"].iloc[0]
    period["running_peak"] = period["wealth_index"].cummax()
    period["drawdown"] = period["wealth_index"] / period["running_peak"] - 1
    return period[["date", "ticker", "wealth_index", "drawdown"]]


def rolling_return_frame(df: pd.DataFrame, start_date: str, end_date: str, window_days: int = 252) -> pd.DataFrame:
    period = slice_period(df, start_date, end_date)
    period["rolling_return"] = period["adj_close"] / period["adj_close"].shift(window_days) - 1
    return period.dropna(subset=["rolling_return"])[["date", "ticker", "rolling_return"]]


def event_impact_window(df: pd.DataFrame, event_date: Union[str, pd.Timestamp], trading_days_before: int = 60, trading_days_after: int = 120) -> pd.DataFrame:
    event_day = row_on_or_after(df, event_date)
    event_index = int(event_day.name)
    start_index = max(0, event_index - trading_days_before)
    end_index = min(len(df) - 1, event_index + trading_days_after)
    window = df.iloc[start_index : end_index + 1].copy()
    event_price = float(event_day["adj_close"])
    window["trading_days_from_event"] = window.index - event_index
    window["relative_return"] = window["adj_close"] / event_price - 1
    window["event_date"] = event_day["date"]
    return window[["date", "ticker", "trading_days_from_event", "relative_return", "event_date"]]


def event_impact_summary(df: pd.DataFrame, event_date: Union[str, pd.Timestamp]) -> dict[str, Any]:
    event_day = row_on_or_after(df, event_date)
    idx = int(event_day.name)
    def rel(day_offset: int) -> Optional[float]:
        pos = idx + day_offset
        if pos < 0 or pos >= len(df):
            return None
        return float(df.iloc[pos]["adj_close"] / event_day["adj_close"] - 1)
    return {
        "event_trading_date": event_day["date"].date().isoformat(),
        "return_30d_before": rel(-30),
        "return_30d_after": rel(30),
        "return_90d_after": rel(90),
        "return_180d_after": rel(180),
        "latest_available_days_after": len(df) - 1 - idx,
        "latest_available_return_after": rel(len(df) - 1 - idx),
    }


stock_wealth = wealth_frame(prices, stock_result["actual_start_date"], stock_result["actual_end_date"], INITIAL_AMOUNT)
benchmark_wealth = wealth_frame(benchmark_prices, benchmark_result["actual_start_date"], benchmark_result["actual_end_date"], INITIAL_AMOUNT)
stock_drawdowns = drawdown_frame(prices, stock_result["actual_start_date"], stock_result["actual_end_date"])
stock_rolling = rolling_return_frame(prices, stock_result["actual_start_date"], stock_result["actual_end_date"])

ceo_event_table = ceo_events_in_period(events, stock_result["actual_start_date"], stock_result["actual_end_date"], limit=8)
story_event_table = ceo_event_table
event_layer_name = "CEO milestones"


### 1. The Outcome

This is the main hero chart: it answers the first user question immediately, then anchors the rest of the story.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.8))
ax.plot(stock_wealth["date"], stock_wealth["wealth"], label=TICKER, linewidth=2.8, color="#2563eb")
ax.plot(benchmark_wealth["date"], benchmark_wealth["wealth"], label=BENCHMARK, linewidth=2.4, color="#52525b")

ax.scatter(stock_wealth.iloc[-1]["date"], stock_wealth.iloc[-1]["wealth"], color="#2563eb", s=70, zorder=3)
ax.scatter(benchmark_wealth.iloc[-1]["date"], benchmark_wealth.iloc[-1]["wealth"], color="#52525b", s=60, zorder=3)
ax.annotate(
    f"{TICKER}: {money(stock_result['final_value'], CURRENCY)}",
    xy=(stock_wealth.iloc[-1]["date"], stock_wealth.iloc[-1]["wealth"]),
    xytext=(-145, 24),
    textcoords="offset points",
    arrowprops={"arrowstyle": "->", "color": "#2563eb"},
    fontsize=10,
    color="#1d4ed8",
)
ax.annotate(
    f"{BENCHMARK}: {money(benchmark_result['final_value'], CURRENCY)}",
    xy=(benchmark_wealth.iloc[-1]["date"], benchmark_wealth.iloc[-1]["wealth"]),
    xytext=(-145, -34),
    textcoords="offset points",
    arrowprops={"arrowstyle": "->", "color": "#52525b"},
    fontsize=10,
    color="#3f3f46",
)

ax.set_title(f"What {money(INITIAL_AMOUNT, CURRENCY)} became", loc="left", fontsize=16, fontweight="bold")
ax.set_ylabel("Portfolio value")
ax.yaxis.set_major_formatter(lambda value, pos: money(value, CURRENCY))
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(frameon=False, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

print(
    f"Reading this chart: {TICKER} compounded to {money(stock_result['final_value'], CURRENCY)}, "
    f"while {BENCHMARK} reached {money(benchmark_result['final_value'], CURRENCY)}. "
    f"The gap is {money(story_result['comparison']['final_value_difference'], CURRENCY)}."
)


### 2. The Hardest Part Of Holding

A good investment story should show the pain along the way. This chart answers: how bad did it feel before the result became obvious?

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.8))
ax.fill_between(stock_drawdowns["date"], stock_drawdowns["drawdown"] * 100, 0, color="#fee2e2")
ax.plot(stock_drawdowns["date"], stock_drawdowns["drawdown"] * 100, color="#dc2626", linewidth=2)

trough_date = pd.Timestamp(drawdown["trough_date"])
trough_row = stock_drawdowns[stock_drawdowns["date"] == trough_date].iloc[0]
ax.scatter(trough_row["date"], trough_row["drawdown"] * 100, color="#991b1b", s=80, zorder=3)
ax.annotate(
    f"Max drawdown: {pct(drawdown['max_drawdown'])}\n{drawdown['peak_date']} to {drawdown['trough_date']}",
    xy=(trough_row["date"], trough_row["drawdown"] * 100),
    xytext=(26, -8),
    textcoords="offset points",
    arrowprops={"arrowstyle": "->", "color": "#991b1b"},
    fontsize=10,
    color="#7f1d1d",
)

ax.set_title(f"The deepest fall while holding {TICKER}", loc="left", fontsize=16, fontweight="bold")
ax.set_ylabel("Drawdown from prior high")
ax.yaxis.set_major_formatter(lambda value, pos: f"{value:.0f}%")
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

print(
    f"Holding through this period required sitting through a {pct(drawdown['max_drawdown'])} drop. "
    f"The recovery date was {drawdown['recovery_date'] or 'outside the selected window'}."
)


### 3. Good Years And Bad Years

This view turns volatility into something understandable: the best and worst roughly one-year stretches inside the holding period.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.8))
ax.axhline(0, color="#71717a", linewidth=1)
ax.plot(stock_rolling["date"], stock_rolling["rolling_return"] * 100, color="#0f766e", linewidth=2)

best = rolling["best"]
worst = rolling["worst"]
for marker, color, label_offset in [(best, "#047857", 18), (worst, "#b91c1c", -42)]:
    if marker:
        row = stock_rolling[stock_rolling["date"] == pd.Timestamp(marker["end_date"])].iloc[0]
        ax.scatter(row["date"], row["rolling_return"] * 100, color=color, s=70, zorder=3)
        ax.annotate(
            f"{pct(marker['return'])}\nending {marker['end_date']}",
            xy=(row["date"], row["rolling_return"] * 100),
            xytext=(18, label_offset),
            textcoords="offset points",
            arrowprops={"arrowstyle": "->", "color": color},
            fontsize=10,
            color=color,
        )

ax.set_title(f"Rolling 12-month return for {TICKER}", loc="left", fontsize=16, fontweight="bold")
ax.set_ylabel("Return over prior 252 trading days")
ax.yaxis.set_major_formatter(lambda value, pos: f"{value:.0f}%")
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

print(
    f"The strongest 12-month stretch returned {pct(best['return'])}; "
    f"the weakest returned {pct(worst['return'])}."
)


### 4. CEO Timeline

This layer is intentionally narrow: only CEO appointments, resignations, and transitions. Other executive changes are usually too noisy for the main investment story.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(stock_wealth["date"], stock_wealth["wealth"], color="#2563eb", linewidth=2, alpha=0.35)

for _, event in story_event_table.iterrows():
    event_date = event["date"]
    nearest = row_on_or_before(stock_wealth, event_date)
    ax.axvline(event_date, color="#a1a1aa", linewidth=1, alpha=0.45)
    ax.scatter(event_date, nearest["wealth"], color="#f59e0b", s=42, zorder=3)

if story_event_table.empty:
    ax.text(
        0.02,
        0.85,
        f"No CEO changes found between {stock_result['actual_start_date']} and {stock_result['actual_end_date']}.",
        transform=ax.transAxes,
        fontsize=11,
        color="#52525b",
    )

for idx, (_, event) in enumerate(story_event_table.head(4).iterrows()):
    nearest = row_on_or_before(stock_wealth, event["date"])
    name = event.get("executive_name") if pd.notna(event.get("executive_name")) else event["event_type"]
    role = event.get("role") if pd.notna(event.get("role")) else ""
    action = event.get("action") if pd.notna(event.get("action")) else ""
    label_parts = [str(name)]
    if action:
        label_parts.append(str(action))
    if role:
        label_parts.append(str(role))
    label = "\n".join(label_parts[:3])
    ax.annotate(
        f"{label}\n{event['date'].date().isoformat()}",
        xy=(event["date"], nearest["wealth"]),
        xytext=(18, 30 if idx % 2 == 0 else -54),
        textcoords="offset points",
        arrowprops={"arrowstyle": "->", "color": "#b45309"},
        fontsize=10,
        color="#92400e",
    )

ax.set_title(f"CEO milestones layered onto the {TICKER} investment path", loc="left", fontsize=16, fontweight="bold")
ax.set_ylabel("Portfolio value")
ax.yaxis.set_major_formatter(lambda value, pos: money(value, CURRENCY))
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

story_event_table


### 5. CEO Event Zoom

The long timeline shows where CEO events happened. This zoom asks the sharper question: what did the stock do around the event date? This is market reaction context, not proof that the event caused the move.

In [ ]:
if story_event_table.empty:
    print("No CEO event inside this selected period, so there is no event-reaction window to zoom into.")
else:
    selected_event = story_event_table.iloc[-1]
    impact = event_impact_window(prices, selected_event["date"], trading_days_before=60, trading_days_after=120)
    impact_summary = event_impact_summary(prices, selected_event["date"])
    event_name = selected_event.get("executive_name") if pd.notna(selected_event.get("executive_name")) else "CEO event"
    event_action = selected_event.get("action") if pd.notna(selected_event.get("action")) else ""

    fig, ax = plt.subplots(figsize=(11, 4.8))
    before = impact[impact["trading_days_from_event"] < 0]
    after = impact[impact["trading_days_from_event"] >= 0]
    ax.plot(before["trading_days_from_event"], before["relative_return"] * 100, color="#94a3b8", linewidth=2, label="Before event")
    ax.plot(after["trading_days_from_event"], after["relative_return"] * 100, color="#2563eb", linewidth=2.5, label="After event")
    ax.axvline(0, color="#b45309", linewidth=2)
    ax.axhline(0, color="#71717a", linewidth=1)
    ax.text(2, ax.get_ylim()[1] * 0.88, "CEO event", color="#92400e", fontsize=10)

    points = [
        (30, impact_summary["return_30d_after"], "30 trading days"),
        (90, impact_summary["return_90d_after"], "90 trading days"),
    ]
    for x, value, label in points:
        if value is None:
            continue
        ax.scatter(x, value * 100, color="#2563eb", s=60, zorder=3)
        ax.annotate(
            f"{label}: {pct(value)}",
            xy=(x, value * 100),
            xytext=(12, 18),
            textcoords="offset points",
            arrowprops={"arrowstyle": "->", "color": "#2563eb"},
            fontsize=10,
            color="#1d4ed8",
        )

    ax.set_title(f"Market reaction around {event_name} {event_action}".strip(), loc="left", fontsize=16, fontweight="bold")
    ax.set_xlabel("Trading days from CEO event")
    ax.set_ylabel("Return vs event-day price")
    ax.yaxis.set_major_formatter(lambda value, pos: f"{value:.0f}%")
    ax.legend(frameon=False, loc="upper left")
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()

    print(
        f"For {event_name}'s CEO event on {impact_summary['event_trading_date']}, "
        f"the stock was {pct(impact_summary['return_30d_after']) if impact_summary['return_30d_after'] is not None else 'not yet 30 trading days available'} after 30 trading days "
        f"and {pct(impact_summary['return_90d_after']) if impact_summary['return_90d_after'] is not None else 'not yet 90 trading days available'} after 90 trading days. "
        f"Latest available reaction: {pct(impact_summary['latest_available_return_after'])} after {impact_summary['latest_available_days_after']} trading days. "
        "This is reaction context, not causality."
    )


## Tables To Inspect

These small tables help us sanity-check the numbers before promoting anything into the engine.

In [ ]:
summary_table = pd.DataFrame([
    {
        "asset": stock_result["ticker"],
        "start": stock_result["actual_start_date"],
        "end": stock_result["actual_end_date"],
        "start_price": stock_result["start_price"],
        "end_price": stock_result["end_price"],
        "final_value": stock_result["final_value"],
        "total_return": stock_result["total_return"],
    },
    {
        "asset": benchmark_result["ticker"],
        "start": benchmark_result["actual_start_date"],
        "end": benchmark_result["actual_end_date"],
        "start_price": benchmark_result["start_price"],
        "end_price": benchmark_result["end_price"],
        "final_value": benchmark_result["final_value"],
        "total_return": benchmark_result["total_return"],
    },
])

summary_table


In [ ]:
event_table = events_in_period(events, stock_result["actual_start_date"], stock_result["actual_end_date"], limit=12)
news_table = recent_news_table(events, limit=5)

event_table


In [ ]:
news_table


## Next Questions Before Moving To `analysis_engine`

1. Should returns use `adj_close` only, or should we explicitly model dividends/splits as cash-flow events?
2. Should inflation be reported in start-year dollars, end-year dollars, or both?
3. Should events be chosen by event importance, proximity to large price moves, or a blend?
4. What is the best cached output shape for frontend cards and charts?

Once this notebook's story feels useful, the next step is to move the stable calculations into `analysis_engine/calculations` and generate cached JSON under `public/data/calculated`.